# Agent Path Analysis (raw per-call trace)

For each question this notebook records the **raw facts** of every LLM call:
- the function-call items the model produced (tool type + `search_query`)
- the full request body sent to the LLM (messages, as-is)
- the final answer

No interpreted labels — you judge the path yourself from the raw data.

Run cells top to bottom. Each question takes ~30-40s (local LLM via the proxy).

## 1. Setup

Builds the hybrid search index (1,368 docs) and the agent from `.env` config.

In [1]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from src.llm_client import LLMClient
from src.rag.prompts import ESCALATION_MESSAGE
from src.rag.rag_agent import RAGAgent
from src.search.hybrid_search import HybridSearch

agent = RAGAgent(search_index=HybridSearch(), llm_client=LLMClient.get())
# The real bound method, captured before any tracing: each trace_run wraps
# the previous wrapper otherwise, and every run's calls leak into all
# earlier logs.
real_call_llm = agent.call_llm
print("confidence threshold:", agent.confidence_threshold)
print("model:", agent.model)

confidence threshold: 0.55
model: qwen/qwen3.5-9b


## 2. Tracing

`trace_run` wraps `agent.call_llm` and records, for **every** call:
- `items`: each function call the model made — `type` (`local` = search_local_knowledge_base, `web` = search_bulbapedia), `name`, and `search_query` (the query argument)
- `escalated`: whether the escalation message (ESCALATION_MESSAGE) was present in the request body before this call
- `request`: a JSON-safe snapshot of the full request body (all messages, including tool outputs)

Returns the final `AgentResult`, the per-call log, and whether escalation fired.

In [2]:
def snapshot_request(messages):
    snap = []
    for m in messages:
        if isinstance(m, dict):
            snap.append({k: v for k, v in m.items()})
        else:
            snap.append({
                "type": "function_call",
                "call_id": getattr(m, "call_id", None),
                "name": getattr(m, "name", None),
                "arguments": getattr(m, "arguments", None),
            })
    return snap


def trace_run(agent, query):
    calls_log = []
    state = {"escalated": False}

    orig = real_call_llm

    def traced(messages, tools=None, temperature=None):
        r = orig(messages, tools=tools, temperature=temperature)
        items = []
        for item in r.output:
            if item.type == "function_call":
                try:
                    args = json.loads(item.arguments or "{}")
                except Exception:
                    args = {}
                items.append({
                    "type": "local" if item.name == "search_local_knowledge_base" else "web",
                    "name": item.name,
                    "search_query": args.get("query", ""),
                })
        escalated_now = any(
            isinstance(m, dict) and m.get("content") == ESCALATION_MESSAGE
            for m in messages
        )
        if escalated_now:
            state["escalated"] = True
        calls_log.append({"items": items, "escalated": escalated_now,
                          "request": snapshot_request(messages)})
        return r

    agent.call_llm = traced
    result = agent.run(query)
    return result, calls_log, state["escalated"]

## 3. Question set

`nature` classifies what the question *needs* (judgment, kept separate from the
raw trace): **local** (answerable from the local KB), **local-partial** (web
needed for completeness), **web** (purely Bulbapedia), **guardrail** (must be
rejected; excluded from the escalation stats).

`expected` is minimal and unprocessed:
- `hybrid -> answer` — first LLM call returns only the hybrid-search tool call, next call returns the answer
- `hybrid -> web -> answer` — hybrid search, then the model calls web on its own, then answers
- `reject` — must be rejected

In [3]:
QUESTIONS = [
    # --- dev-subset questions (local KB) ---
    {"question": "What are Bulbasaur's two main types and how does that affect its weaknesses to fire and ice moves?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "If I want to catch this Seed Pokémon in the wild, what is its capture rate compared to other species?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Which ability acts as Bulbasaur's hidden ability besides Overgrow, and when would it be most useful?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "How strong are Grass-type attacks against Bulbasaur given its specific type effectiveness ratios?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Ivysaur is a Grass and Poison type, so does it take double damage from Ice moves?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Since Ivysaur has the Overgrow ability, what happens if its HP gets really low in battle?", "nature": "local-partial", "expected": "hybrid -> answer"},
    {"question": "If I want to evolve Ivysaur into Venusaur, do I need any specific items or just level it up?", "nature": "local-partial", "expected": "hybrid -> answer"},
    {"question": "Ivysaur is a quadruped shape with green coloring, but does that mean it can't stand on two legs like Bulbasaur?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "With a base stat total of 405 and Chlorophyll as its hidden ability, how fast does Ivysaur move compared to other Pokémon?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Does Venusaur have a hidden ability, and if so, what is it?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "What are the specific stats for Venusaur's special attack and defense?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Which types of moves deal super effective damage to this Grass/Poison type Pokémon?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Is Venusaur capable of evolving into another form or is it already at its final stage?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "What kind of habitat does Venusaur naturally live in according to its profile?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "What is Charmander's base stat total and how does its speed compare to its attack?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Which ability becomes hidden for this Lizard Pokémon when it has high sunlight exposure?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "If I try to catch a wild Charmander in the mountains, what are my odds of success based on its capture rate?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "How effective is Water-type moves against Charmander compared to Fire or Rock types?", "nature": "local", "expected": "hybrid -> answer"},
    {"question": "Does this baby Pokémon evolve into something legendary or does it belong to a different evolution chain?", "nature": "local", "expected": "hybrid -> answer"},
    # --- web-only (Bulbapedia) ---
    {"question": "Who voiced Pikachu in the anime?", "nature": "web", "expected": "hybrid -> web -> answer"},
    {"question": "What is the newest Pokémon introduced in Scarlet/Violet?", "nature": "web", "expected": "hybrid -> web -> answer"},
    # --- guardrail (must reject; excluded from escalation analysis) ---
    {"question": "Who won the 2024 Super Bowl?", "nature": "guardrail", "expected": "reject"},
    {"question": "asdfghjkl", "nature": "guardrail", "expected": "reject"},
    {"question": "Tell me about Abraham Lincoln", "nature": "guardrail", "expected": "reject"},
]

## 4. Run

For each question, print the raw per-call record: call items (type +
`search_query` pairs), `[ESC]` when the escalation message was in that call's
request body, the request body of each call (roles/contents, tool outputs
truncated in this print — the full bodies stay in `trace_results`), and the
final answer.

In [4]:
trace_results = {}

for i, q in enumerate(QUESTIONS, 1):
    result, calls, escalated = trace_run(agent, q["question"])
    trace_results[i] = {"question": q["question"], "result": result, "calls": calls}
    print(f"=== [{i}] {q['nature']} | expected: {q['expected']} ===")
    print(f"Q: {q['question']}")
    for j, c in enumerate(calls, 1):
        items = ", ".join(f"({it['type']}, query={it['search_query']!r})" for it in c["items"]) or "no tool call"
        esc = " [ESC]" if c["escalated"] else ""
        body = " | ".join(
            (m.get("role") or m.get("type") or "?")
            + ":" + (str(m.get("content") or m.get("output") or m.get("arguments") or "")[:70])
            for m in c["request"]
        )
        print(f"  call{j}: items=[{items}]{esc}")
        print(f"    request: {body}")
    status = "rejected" if result.rejected else "accepted"
    print(f"  -> {status}, source={result.source}, confidence={result.confidence and round(result.confidence, 3)}, relevance={result.relevance and round(result.relevance, 3)}")
    print(f"  answer: {(result.answer or '')[:300]}")
    print()

=== [1] local | expected: hybrid -> answer ===
Q: What are Bulbasaur's two main types and how does that affect its weaknesses to fire and ice moves?
  call1: items=[(local, query='Bulbasaur types weaknesses')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What are Bulbasaur's two main types and how does that affect its weakn
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What are Bulbasaur's two main types and how does that affect its weakn | function_call:{"query":"Bulbasaur types weaknesses"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  -> accepted, source=local, confidence=0.616, relevance=0.865
  answer: Bulbasaur's two main types are **Grass** and **Poison**.

This dual typing affects its weaknesses to Fire and Ice moves in the following ways:

*   **Fire Moves:** Bulbasaur is **weak** (takes 2

=== [2] local | expected: hybrid -> answer ===
Q: If I want to catch this Seed Pokémon in the wild, what is its capture rate compared to other species?
  call1: items=[(local, query='Seed Pokémon capture rate')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:If I want to catch this Seed Pokémon in the wild, what is its capture 
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:If I want to catch this Seed Pokémon in the wild, what is its capture  | function_call:{"query":"Seed Pokémon capture rate"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  -> accepted, source=local, confidence=0.639, relevance=0.629
  answer: Based on the retrieved data, there are a few Pokémon with "Seed" in their genus name (Bulbasaur and Venusaur forms), but only one Pokémon whose name literally starts with "Seed": **Seedot**.

H

=== [3] local | expected: hybrid -> answer ===
Q: Which ability acts as Bulbasaur's hidden ability besides Overgrow, and when would it be most useful?
  call1: items=[(local, query='Bulbasaur abilities hidden ability')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Which ability acts as Bulbasaur's hidden ability besides Overgrow, and
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Which ability acts as Bulbasaur's hidden ability besides Overgrow, and | function_call:{"query":"Bulbasaur abilities hidden ability"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  -> accepted, source=local, confidence=0.577, relevance=0.758
  answer: Bulbasaur's hidden ability is **Chlorophyll**.

It would be most useful during the day in battles or when playing on a sunny field (in games like *Pokémon Black/White* and late

=== [4] local | expected: hybrid -> answer ===
Q: How strong are Grass-type attacks against Bulbasaur given its specific type effectiveness ratios?
  call1: items=[(local, query='Bulbasaur types weaknesses')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:How strong are Grass-type attacks against Bulbasaur given its specific
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:How strong are Grass-type attacks against Bulbasaur given its specific | function_call:{"query":"Bulbasaur types weaknesses"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  -> accepted, source=local, confidence=0.683, relevance=0.875
  answer: Based on the retrieved data, Bulbasaur is a dual-type Pokémon with **Grass** and **Poison** typing. The type effectiveness ratios for Grass-type attacks against Bulbasaur are calculated as follow

=== [5] local | expected: hybrid -> answer ===
Q: Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?
  call1: items=[(local, query='Bulbasaur evolution')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Does Bulbasaur evolve into a later stage since the data shows it is no
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Does Bulbasaur evolve into a later stage since the data shows it is no | function_call:{"query":"Bulbasaur evolution"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  call3: items=[no tool call] [ESC]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Does Bulbasaur evolve into a later stage since the data shows it is no | function_call:{"query":"Bulbasaur evolution"} | function_call_output

...

=== [21] web | expected: hybrid -> web -> answer ===
Q: Who voiced Pikachu in the anime?
  call1: items=[(local, query='Pikachu voice actor anime')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who voiced Pikachu in the anime?
  call2: items=[(web, query='Pikachu voice actor anime')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who voiced Pikachu in the anime? | function_call:{"query":"Pikachu voice actor anime"} | function_call_output:[{"name": "Pikachu-Gmax", "text": "Pokémon: Pikachu-Gmax (#10199)\nGen
  call3: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who voiced Pikachu in the anime? | function_call:{"query":"Pikachu voice actor anime"} | function_call_output:[{"name": "Pikachu-Gmax", "text": "Pokémon: Pikachu-Gmax (#10199)\nGen | function_call:{"query":"Pikachu voice actor anime"} | function_

=== [22] web | expected: hybrid -> web -> answer ===
Q: What is the newest Pokémon introduced in Scarlet/Violet?
  call1: items=[(local, query='newest Pokémon Scarlet Violet introduction')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What is the newest Pokémon introduced in Scarlet/Violet?
  call2: items=[(web, query='newest Pokémon Scarlet Violet introduction Paradox forms')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What is the newest Pokémon introduced in Scarlet/Violet? | function_call:{"query":"newest Pokémon Scarlet Violet introduction"} | function_call_output:[{"name": "Minior-Violet", "text": "Pokémon: Minior-Violet (#10142)\nG
  call3: items=[(web, query='newest Pokémon Scarlet Violet Paradox forms Walking Wake Iron Leaves Dipplin Archaludon Raging Bolt Iron Crown')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | u

=== [23] guardrail | expected: reject ===
Q: Who won the 2024 Super Bowl?
  call1: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who won the 2024 Super Bowl?
  -> rejected, source=None, confidence=None, relevance=None
  answer: I'm a Pokémon knowledge assistant — I can answer questions about Pokémon stats, types, weaknesses, abilities, evolutions, and type matchups. I can't predict battle outcomes, access save files, help with cheating, or answer non-Pokémon topics. Try asking about a specific Pokémon!



=== [24] guardrail | expected: reject ===
Q: asdfghjkl
  call1: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:asdfghjkl
  call2: items=[no tool call] [ESC]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:asdfghjkl | user:Your previous answer was not grounded in the retrieved documents. Call
  -> rejected, source=None, confidence=None, relevance=None
  answer: I'm a Pokémon knowledge assistant — I can answer questions about Pokémon stats, types, weaknesses, abilities, evolutions, and type matchups. I can't predict battle outcomes, access save files, help with cheating, or answer non-Pokémon topics. Try asking about a specific Pokémon!



=== [25] guardrail | expected: reject ===
Q: Tell me about Abraham Lincoln
  call1: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Tell me about Abraham Lincoln
  -> rejected, source=None, confidence=None, relevance=None
  answer: I'm a Pokémon knowledge assistant — I can answer questions about Pokémon stats, types, weaknesses, abilities, evolutions, and type matchups. I can't predict battle outcomes, access save files, help with cheating, or answer non-Pokémon topics. Try asking about a specific Pokémon!



## 5. Summary (derived — raw data is in section 4)

Escalation rate and per-nature outcomes. Guardrail questions are excluded from
the escalation stats.

In [5]:
import pandas as pd

rows = []
for i, q in enumerate(QUESTIONS, 1):
    t = trace_results[i]
    r = t["result"]
    n_web = sum(
        1 for c in t["calls"] for it in c["items"] if it["type"] == "web"
    )
    rows.append({
        "#": i,
        "question": q["question"],
        "nature": q["nature"],
        "expected": q["expected"],
        "llm calls": len(t["calls"]),
        "escalated": any(c["escalated"] for c in t["calls"]),
        "web calls (any)": n_web,
        "rejected": r.rejected,
        "source": r.source,
        "confidence": round(r.confidence, 3) if r.confidence else None,
        "relevance": round(r.relevance, 3) if r.relevance is not None else None,
    })

df = pd.DataFrame(rows)
df

,#,question,nature,expected,llm calls,escalated,web calls (any),rejected,source,confidence,relevance
0,1,What are Bulbasaur's two main types and how do...,local,hybrid -> answer,2,False,0,False,local,0.616,0.865
1,2,If I want to catch this Seed Pokémon in the wi...,local,hybrid -> answer,2,False,0,False,local,0.639,0.629
2,3,Which ability acts as Bulbasaur's hidden abili...,local,hybrid -> answer,2,False,0,False,local,0.577,0.758
3,4,How strong are Grass-type attacks against Bulb...,local,hybrid -> answer,2,False,0,False,local,0.683,0.875
4,5,Does Bulbasaur evolve into a later stage since...,local,hybrid -> answer,3,True,0,False,local,0.596,0.908
5,6,"Ivysaur is a Grass and Poison type, so does it...",local,hybrid -> answer,3,True,0,True,NaN,NaN,NaN
6,7,"Since Ivysaur has the Overgrow ability, what h...",local-partial,hybrid -> answer,4,True,1,False,local+web,0.623,0.776
7,8,"If I want to evolve Ivysaur into Venusaur, do ...",local-partial,hybrid -> answer,4,True,1,False,local+web,0.718,0.869
8,9,Ivysaur is a quadruped shape with green colori...,local,hybrid -> answer,3,True,0,True,NaN,NaN,NaN
9,10,With a base stat total of 405 and Chlorophyll ...,local,hybrid -> answer,2,False,0,False,local,0.594,0.851


In [6]:
main = df[df["nature"] != "guardrail"]
n = len(main)
n_esc = main["escalated"].sum()
n_web_self = ((main["web calls (any)"] > 0) & (~main["escalated"])).sum()
print(f"in-scope questions: {n}")
print(f"escalated: {n_esc} ({n_esc / n:.0%})")
print(f"web searched WITHOUT escalation (model-initiated): {n_web_self}")
print()
print("per-nature outcomes:")
print(main.groupby("nature")["rejected"].agg(["count", "sum"]).rename(columns={"count": "n", "sum": "rejected"}))
print()
print("guardrail (must reject):")
print(df[df["nature"] == "guardrail"][["#", "rejected", "source"]].to_string(index=False))

in-scope questions: 22
escalated: 11 (50%)
web searched WITHOUT escalation (model-initiated): 2

per-nature outcomes:
                n  rejected
nature                     
local          18         6
local-partial   2         0
web             2         0

guardrail (must reject):
 #  rejected source
23      True    NaN
24      True    NaN
25      True    NaN


In [7]:
def expected_ok(row):
    if row["expected"] == "reject":
        return bool(row["rejected"])
    if row["expected"] == "hybrid -> web -> answer":
        return (not row["escalated"]) and row["web calls (any)"] > 0 and not row["rejected"]
    return (not row["escalated"]) and row["web calls (any)"] == 0 and not row["rejected"]


def abnormal_reasons(row):
    reasons = []
    if row["expected"] == "reject":
        if not row["rejected"]:
            reasons.append("accepted (expected reject)")
    else:
        if row["escalated"]:
            reasons.append("escalation fired")
        if row["rejected"]:
            reasons.append("rejected (answer failed grounding)")
        if row["web calls (any)"] > 0 and "web" not in row["expected"]:
            reasons.append("web called (expected local only)")
        if row["web calls (any)"] == 0 and "web" in row["expected"]:
            reasons.append("web not called (expected model-initiated web)")
    return "; ".join(reasons) or "ok"


abnormal = df[~df.apply(expected_ok, axis=1)].copy()
abnormal["reasons"] = abnormal.apply(abnormal_reasons, axis=1)
print(f"abnormal: {len(abnormal)} of {len(df)} — actual path differs from expected\n")
for _, r in abnormal.iterrows():
    print(f"[{r['#']:2d}] {r['nature']:13s} | expected {r['expected']:22s} | calls={r['llm calls']} "
          f"| esc={str(r['escalated']):5s} web={r['web calls (any)']} rejected={str(r['rejected']):5s} "
          f"| source={r['source']} | conf={r['confidence']} rel={r['relevance']}")
    print(f"     Q: {r['question']}")
    print(f"     why: {r['reasons']}

abnormal: 11 of 25 — actual path differs from expected

[ 5] local         | expected hybrid -> answer       | calls=3 | esc=True  web=0 rejected=False | source=local | conf=0.596 rel=0.908
     Q: Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?
     why: escalation fired
[ 6] local         | expected hybrid -> answer       | calls=3 | esc=True  web=0 rejected=True  | source=nan | conf=nan rel=nan
     Q: Ivysaur is a Grass and Poison type, so does it take double damage from Ice moves?
     why: escalation fired; rejected (answer failed grounding)
[ 7] local-partial | expected hybrid -> answer       | calls=4 | esc=True  web=1 rejected=False | source=local+web | conf=0.623 rel=0.776
     Q: Since Ivysaur has the Overgrow ability, what happens if its HP gets really low in battle?
     why: escalation fired; web called (expected local only)
[ 8] local-partial | expected hybrid -> answer       | calls=4 | esc=True  web=1 rejected=False